# Week 28, Day 1: Why objects?

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w028d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
name = "ada lovelace"
scores = [72, 91, 64]
print(type(name), type(scores))
print(name.title(), name)
scores.append(80)
print(scores)

**Predict before you run:** what will this print?



In [ ]:
words = ["pear", "fig"]
print(type(words).__name__, type(3).__name__, type(len).__name__, type(None).__name__)
print([m for m in dir(words) if not m.startswith("_")][:6])
print(isinstance(words, list), isinstance(True, int))

**Predict before you run:** what will this print?



In [ ]:
a = [1, 2]
b = a
c = [1, 2]
print(a == c, a is c, a is b)
b.append(3)
print(a, c)

> **Key idea.** An object has state (its data), behaviour (its methods) and identity (which one it is). Its type, or class, decides its methods. Everything in Python is an object, and `type`, `dir`, `isinstance`, `is` and `id` let you look at them.

In [ ]:
def make_account(owner, balance=0):
    return {"owner": owner, "balance": balance, "history": []}

def deposit(account, amount):
    if amount <= 0:
        raise ValueError("deposit must be positive")
    account["balance"] += amount
    account["history"].append(("deposit", amount))

def withdraw(account, amount):
    if amount > account["balance"]:
        raise ValueError("not enough money")
    account["balance"] -= amount
    account["history"].append(("withdraw", amount))

acc = make_account("Ada", 100)
deposit(acc, 50)
withdraw(acc, 30)
print(acc["balance"], acc["history"])

acc["balance"] = -1_000_000          # nothing stops this
acc["balence"] = 5                   # or this typo, which quietly makes a new key
print(acc)

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance
        self.history = []

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.balance += amount
        self.history.append(("deposit", amount))

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("not enough money")
        self.balance -= amount
        self.history.append(("withdraw", amount))

ada = Account("Ada", 100)
bo = Account("Bo")
ada.deposit(50)
ada.withdraw(30)
bo.deposit(5)
print(ada.owner, ada.balance, ada.history)
print(bo.owner, bo.balance)
print(type(ada).__name__, isinstance(ada, Account))

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance
        self.history = []

ada = Account("Ada", 100)
bo = Account("Bo")
ada.history.append(("deposit", 50))
print(vars(ada))
print(vars(bo))
print(ada is bo, ada.history is bo.history)

> **Key idea.** A class bundles data (attributes) with the functions that work on it (methods), and names the shape. Instances are made by calling the class; `__init__` sets up their attributes; methods get the instance as `self`. To design with objects, find the nouns, what they know and what they do. Use classes when data and rules belong together, not everywhere.

### Exercise: What type is it?

Give the type name (as `type(x).__name__` would show it) of each value.

In [ ]:
a = ___   # "42"
b = ___   # 42 / 2
c = ___   # [1, 2] == [1, 2]
d = ___   # {"a": 1}.items()
e = ___   # print
f = ___   # range(3)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (a, b, c, d, e, f) == (\"str\", \"float\", \"bool\", \"dict_items\", \"builtin_function_or_method\", \"range\")")

### Exercise: Same object, or equal?

Predict each, then check.

In [ ]:
x = [1, 2, 3]
y = x
z = list(x)
y.append(4)

a = ___   # x == z
b = ___   # x is y
c = ___   # x is z
d = ___   # len(z)
e = ___   # x == y

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (a, b, c, d, e) == (False, True, False, 3, True)")

### Exercise: Change it, or make a new one?

For each method, write `"changes"` if it changes the object it's called on, or `"new"` if it leaves it alone and returns a new value.

In [ ]:
a = ___   # "text".upper()
b = ___   # [3, 1].sort()
c = ___   # sorted([3, 1])
d = ___   # {"a": 1}.update({"b": 2})
e = ___   # "a,b".split(",")
f = ___   # [1].extend([2])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (a, b, c, d, e, f) == (\"new\", \"changes\", \"new\", \"changes\", \"new\", \"changes\")")

### Exercise: Use a class

Using the `Account` class in the starter, make an account for `"Tola"` with 200, deposit 75, withdraw 120, and make a second account for `"Sam"` with nothing in it. Then print Tola's balance and Sam's balance on one line, separated by a space.

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance
        self.history = []

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.balance += amount
        self.history.append(("deposit", amount))

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("not enough money")
        self.balance -= amount
        self.history.append(("withdraw", amount))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the balances\"\n    expect_output(\"155 0\")\ndef test_objects():\n    \"two accounts, used through their methods\"\n    accounts = [v for v in ns.values() if isinstance(v, ns[\"Account\"])]\n    assert len(accounts) == 2, \"make two Account instances, in variables\"\n    tola = next(a for a in accounts if a.owner == \"Tola\")\n    assert tola.history == [(\"deposit\", 75), (\"withdraw\", 120)]")

### Exercise: Add a method

Add a method `statement(self)` to `Account` that returns a list of lines: the owner's name, then one line per history entry like `deposit 75` or `withdraw 120`, then `balance 155`. Copy the pattern of the existing methods: it's indented inside the class, and its first parameter is `self`.

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance
        self.history = []

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.balance += amount
        self.history.append(("deposit", amount))

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("not enough money")
        self.balance -= amount
        self.history.append(("withdraw", amount))

    # write statement here

tola = Account("Tola", 200)
tola.deposit(75)
tola.withdraw(120)
print("\n".join(tola.statement()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"Tola's statement\"\n    expect_output(\"Tola\\ndeposit 75\\nwithdraw 120\\nbalance 155\")\ndef test_new():\n    \"a new account\"\n    a = ns[\"Account\"](\"Ade\", 9)\n    assert a.statement() == [\"Ade\", \"balance 9\"]")

### Exercise: Find the classes

Read this description, then fill in the model as a dictionary from class names to a pair `(attributes, methods)`, each a set of lower-case names from the description.

> A gym has **members** and **classes** (sessions, not Python classes!). A member has a name and a membership level. A session has a title, a time and a capacity, and keeps a list of bookings. A member can book a session and cancel a booking. A session can tell you how many places are left.

Use the class names `Member` and `Session`; attributes `name`, `level`, `title`, `time`, `capacity`, `bookings`; and methods `book`, `cancel`, `places_left`.

In [ ]:
model = {
    "Member": (___, ___),
    "Session": (___, ___),
}

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_model():\n    \"the model\"\n    assert model == {\"Member\": ({\"name\", \"level\"}, {\"book\", \"cancel\"}),\n                     \"Session\": ({\"title\", \"time\", \"capacity\", \"bookings\"}, {\"places_left\"})}")

### Bug hunt: Two names, one list

This should keep two separate shopping lists, but adding to Bo's also adds to Ada's. Fix it so each starts as its own empty list.

In [ ]:
ada = bo = []
ada.append("milk")
bo.append("eggs")
print(ada, bo)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"separate lists\"\n    expect_output(\"['milk'] ['eggs']\")")

### Bug hunt: Forgot the instance

Calling `counter.add()` crashes with `TypeError: Counter.add() takes 0 positional arguments but 1 was given`. Python passes the instance to every method. Fix the class so the program prints `3`.

In [ ]:
class Counter:
    def __init__(self):
        self.count = 0

    def add():
        self.count += 1

counter = Counter()
counter.add()
counter.add()
counter.add()
print(counter.count)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"counts\"\n    expect_output(\"3\")")

### Puzzle: A tiny class

Arrange the lines into a class `Lamp` that starts off and can be switched, and a program that switches it twice and prints `False`. One line is a trap.

(Put these lines in order on the website.)

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a tally

Write a class `Tally` for counting votes. `Tally()` starts empty. `vote(name)` adds a vote for `name` (stripped and in title case: `" ada "` counts for `"Ada"`); an empty name raises `ValueError`. `count(name)` gives the votes for a name (0 if none). `winner()` returns the name with most votes, or `None` with no votes; a tie goes to the name that **reached** the top count first. `results()` returns `(name, votes)` pairs, most votes first, A to Z for ties.

In [ ]:
class Tally:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_votes():\n    \"voting and counting\"\n    t = Tally()\n    for v in [\"ada\", \" Bo \", \"ADA\", \"cy\", \"bo\"]:\n        t.vote(v)\n    assert t.count(\"Ada\") == 2 and t.count(\"Bo\") == 2 and t.count(\"Zed\") == 0\n    assert t.results() == [(\"Ada\", 2), (\"Bo\", 2), (\"Cy\", 1)]\ndef test_winner():\n    \"winner\"\n    t = Tally()\n    assert t.winner() is None\n    for v in [\"Bo\", \"Ada\", \"Ada\", \"Bo\"]:\n        t.vote(v)\n    assert t.winner() == \"Ada\"\n    t.vote(\"Bo\")\n    assert t.winner() == \"Bo\"\ndef test_separate():\n    \"each tally is separate, and empty names are refused\"\n    a, b = Tally(), Tally()\n    a.vote(\"x\")\n    assert b.count(\"X\") == 0 and b.results() == []\n    try:\n        a.vote(\"   \")\n        assert False, \"an empty name should raise ValueError\"\n    except ValueError:\n        pass")

## Warm-up

`name.title()` returns a **new** string and leaves `name` alone, while `scores.append(80)` **changes** the list itself. Both are **methods**: functions that belong to a type of value and are called with a dot. Strings and lists are **objects**, and so is everything else in Python.

## You've used objects all along

An **object** is a value with three things:

- **state**: the data it holds (the characters of a string; the items in a list);
- **behaviour**: the methods you can call on it (`upper`, `split`, `append`, `sort`);
- **identity**: which particular object it is, even when another one looks the same.

Every object has a **type** (also called its **class**), which decides what methods it has. `type(x)` tells you the type; `dir(x)` lists everything it has; `help(x.method)` explains a method.

Even functions, `None` and whole numbers are objects. `isinstance(True, int)` is True because `bool` is a special kind of `int`, which you'll understand properly in Week 30.

**Identity** is what `is` checks: whether two names refer to the **same** object, not just equal ones. `id(x)` gives a number that's unique to the object while it exists.

`a` and `b` are two names for one list, so changing it through `b` shows through `a`. `c` is a different list that happened to be equal. You met this aliasing in Phase 1; with objects of your own it matters even more.

## When data and functions drift apart

Here's how you might keep track of bank accounts with what you know so far: dictionaries for the data, and functions that take a dictionary.

It works, and in a small program it's fine. But as programs grow, problems appear:

- **Nothing protects the data.** Any code anywhere can set the balance to anything, skipping every check `withdraw` makes. A typo in a key quietly creates a new key.
- **The rules are scattered.** To learn what an account can do, you have to find every function that takes one, across every file.
- **The shape is a convention.** Nothing says an account must have `"owner"`, `"balance"` and `"history"`; every function just hopes.
- **Names clash.** A `deposit` for accounts and a `deposit` for a savings jar can't both be called `deposit`.

You met these problems in Week 25: spend's expenses were dictionaries with an agreed shape, and every function had to trust it. A **class** puts the data and the functions that work on it in one place, and gives the shape a name.

## A first class

Here's the same account as a class. Don't worry about every detail yet; the next three days explain each part.

What to notice:

- `class Account:` defines a new **type**. `Account("Ada", 100)` makes an **instance** of it: one particular account. `ada` and `bo` are two instances, each with its own state.
- `__init__` runs when an instance is made, and sets up its **attributes**: `self.owner`, `self.balance`, `self.history`.
- `deposit` and `withdraw` are **methods**, defined inside the class. You call them with a dot, `ada.deposit(50)`, just like `scores.append(80)`.
- `self` is the instance the method was called on. In `ada.deposit(50)`, `self` is `ada`.

The data and the rules now live together. Everything an account can do is listed in one place, and `ada.withdraw` can only ever mean the account version. (Python still lets you write `ada.balance = -1000000` directly; Week 29 shows how to protect attributes properly.)

Each instance keeps its attributes in a dictionary of its own, which `vars` shows you. That's another way to see that `ada` and `bo` are separate:

## Thinking in objects

Designing with objects starts with the problem, not the code. A simple, effective technique: read the description and

- underline the **nouns**: they're candidate **classes** (and some are attributes);
- note what each one **knows**: its attributes;
- note what each one **does**, or has done to it: the **verbs**, its methods.

> A library lends **books** to **members**. Each book has a title, an author and a shelf mark, and can be on loan or available. A member has a name and a card number, can borrow up to five books, and returns them. The library records each **loan** with its due date, and charges a fine when a book comes back late.

| Class | Knows (attributes) | Does (methods) |
|---|---|---|
| `Book` | title, author, shelf mark, whether it's on loan | |
| `Member` | name, card number, books on loan | borrow, give back |
| `Loan` | book, member, due date, returned date | is it late? work out the fine |
| `Library` | books, members, loans | lend, take back, find a book |

Not every noun becomes a class: a title is just a string, a due date is a `date`. Make a class when something has **several pieces of data that belong together** and **rules about them**.

And not everything should be a class. A function that turns Celsius into Fahrenheit doesn't need one; neither does a short script. Classes earn their place when there are several things of the same kind, each with its own state, and rules that must always be kept.

## Practice

## Quiz

## Daily challenge

## Before you go

Take spend from Week 25, and sketch it as classes on paper: what would an `Expense` know and do? What about the collection of expenses with its budgets and next id? Keep the sketch: by the end of this phase you'll be able to build it. Tomorrow: classes and instances properly, from the `class` statement up.